In [1]:
#  Get input data for time of scan
import os
import re
from datetime import datetime, timedelta
import scipy.io
import meme.archive

In [7]:
inputs = [
    "CAMR:IN20:186:XRMS",
    "CAMR:IN20:186:YRMS",
    "SOLN:IN20:121:BCTRL",
    "QUAD:IN20:121:BCTRL",
    "QUAD:IN20:122:BCTRL",
    "ACCL:IN20:300:L0A_ADES",
    "ACCL:IN20:300:L0A_PDES",
    "ACCL:IN20:400:L0B_PDES",
    "QUAD:IN20:361:BCTRL",
    "QUAD:IN20:371:BCTRL",
    "QUAD:IN20:425:BCTRL",
    "QUAD:IN20:441:BCTRL",
    "QUAD:IN20:511:BCTRL",
    "QUAD:IN20:525:BCTRL"
]

In [8]:
scans_dir = f"matlab_scans/"
regex = r"^Emittance-scan-OTRS_IN20_541-2022-[0-9]{2}-[0-9]{2}-[0-9]{6}\.mat$"


filenames=[]
for file in os.scandir(scans_dir):
    if re.match(regex, file.name):
        filenames.append(file.name)
print('Total ',len(filenames),'files')

Total  9 files


In [9]:
fname = 'Emittance-scan-OTRS_IN20_541-2022-01-25-055155.mat'

def matlab_datenum_to_datetime(datenum):
    datenum = float(datenum)
    return datetime.fromordinal(int(datenum)) + timedelta(days=datenum % 1) - timedelta(days=366)

mat_data = scipy.io.loadmat('matlab_scans/' + fname)['data'][0]
ts = mat_data['ts'][0][0].item()  
ts = matlab_datenum_to_datetime(ts).strftime('%Y-%m-%d %H:%M:%S')

In [10]:
d_files = {f: {} for f in filenames}

for fname in filenames:
    mat_data = scipy.io.loadmat('matlab_scans/' + fname)['data'][0]
    ts = mat_data['ts'][0][0].item()  
    ts = matlab_datenum_to_datetime(ts).strftime('%Y-%m-%d %H:%M:%S')
    results = meme.archive.get(inputs, from_time=ts, to_time=ts)
    
    #d_files[fname] = {i: [] for i in inputs}
    for pv in results:
        d_files[fname][pv["pvName"]] = (pv["value"]["value"]["values"].item())

    print(d_files)

{'Emittance-scan-OTRS_IN20_541-2022-01-25-055155.mat': {'CAMR:IN20:186:XRMS': 307.1037298034882, 'CAMR:IN20:186:YRMS': 361.3634461387238, 'SOLN:IN20:121:BCTRL': 0.4537619, 'QUAD:IN20:121:BCTRL': 0.0034535998, 'QUAD:IN20:122:BCTRL': -8.8532147e-05, 'ACCL:IN20:300:L0A_ADES': 58.0, 'ACCL:IN20:300:L0A_PDES': 0.0, 'ACCL:IN20:400:L0B_PDES': -2.5, 'QUAD:IN20:361:BCTRL': -3.3422473696965262, 'QUAD:IN20:371:BCTRL': 2.6437607244285766, 'QUAD:IN20:425:BCTRL': 1.0, 'QUAD:IN20:441:BCTRL': -0.7677491240231058, 'QUAD:IN20:511:BCTRL': 3.2440699650318114, 'QUAD:IN20:525:BCTRL': -3.045434069243833}, 'Emittance-scan-OTRS_IN20_541-2022-01-23-012229.mat': {}, 'Emittance-scan-OTRS_IN20_541-2022-01-26-030852.mat': {}, 'Emittance-scan-OTRS_IN20_541-2022-01-23-014920.mat': {}, 'Emittance-scan-OTRS_IN20_541-2022-01-23-011437.mat': {}, 'Emittance-scan-OTRS_IN20_541-2022-01-23-021526.mat': {}, 'Emittance-scan-OTRS_IN20_541-2022-01-23-021311.mat': {}, 'Emittance-scan-OTRS_IN20_541-2022-01-25-055404.mat': {}, 'Emit

In [12]:
import json

with open("pv_data.json", "w") as f:
    json.dump(d_files, f, indent=4) 